In [14]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

env = gym.make('Blackjack-v1', sab=True)

In [15]:
def generate_episode(env, policy):
    episode = []
    observation, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = policy(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        episode.append((observation, action, reward))
        observation = next_observation
    return episode

In [16]:
def epsilon_greedy_policy(Q, observation, epsilon):
    num_actions = env.action_space.n

    if np.random.rand() < epsilon:
        return np.random.randint(num_actions)
    else:
        return np.argmax(Q[observation])

In [17]:
def mc_on_policy_control(env, num_episodes, epsilon_start, epsilon_end, epsilon_decay):
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    returns_sum = defaultdict(lambda: np.zeros(env.action_space.n))
    returns_count = defaultdict(lambda: np.zeros(env.action_space.n))

    for i_episode in range(1, num_episodes + 1):
        epsilon = max(epsilon_end, epsilon_start - epsilon_decay * i_episode)

        episode = []
        observation, info = env.reset()
        terminated = False
        truncated = False

        while not terminated and not truncated:
            action = epsilon_greedy_policy(Q, observation, epsilon)
            next_observation, reward, terminated, truncated, info = env.step(action)
            episode.append((observation, action, reward))
            observation = next_observation

        states, actions, rewards = zip(*episode)

        sa_in_episode = set()
        for i, (state, action) in enumerate(zip(states, actions)):
            if (state, action) not in sa_in_episode:
                sa_in_episode.add((state, action))
                G = sum(rewards[i:])
                returns_sum[state][action] += G
                returns_count[state][action] += 1.0
                Q[state][action] = returns_sum[state][action] / returns_count[state][action]

    return Q

In [18]:
num_episodes_mc = 50000
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = (epsilon_start - epsilon_end) / num_episodes_mc

Q_optimal = mc_on_policy_control(env, num_episodes_mc, epsilon_start, epsilon_end, epsilon_decay)

def get_action_name(action):
    return "Stick" if action == 0 else "Hit"

state1 = (18, 7, False)
if state1 in Q_optimal:
    print(f"State: {state1}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state1][action]:.4f}")

state2 = (12, 2, False)
if state2 in Q_optimal:
    print(f"State: {state2}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state2][action]:.4f}")

state3 = (20, 5, False)
if state3 in Q_optimal:
    print(f"State: {state3}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state3][action]:.4f}")

state4 = (17, 1, True)
if state4 in Q_optimal:
    print(f"State: {state4}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state4][action]:.4f}")

state5 = (19, 8, True)
if state5 in Q_optimal:
    print(f"State: {state5}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state5][action]:.4f}")

State: (18, 7, False)
  Action: Stick, Q-Value: 0.4053
  Action: Hit, Q-Value: -0.6931
State: (12, 2, False)
  Action: Stick, Q-Value: -0.2443
  Action: Hit, Q-Value: -0.4071
State: (20, 5, False)
  Action: Stick, Q-Value: 0.6389
  Action: Hit, Q-Value: -0.9621
State: (17, 1, True)
  Action: Stick, Q-Value: -0.5952
  Action: Hit, Q-Value: -0.7241
State: (19, 8, True)
  Action: Stick, Q-Value: 0.4833
  Action: Hit, Q-Value: -0.1250


In [19]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

# Create an instance of the Blackjack environment
env = gym.make('Blackjack-v1', sab=True)

print("Blackjack environment initialized.")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")

Blackjack environment initialized.
Observation Space: Tuple(Discrete(32), Discrete(11), Discrete(2))
Action Space: Discrete(2)


In [20]:
Q = defaultdict(lambda: np.zeros(env.action_space.n))
returns_sum = defaultdict(float)
returns_count = defaultdict(float)

print("Q-values, returns_sum, and returns_count initialized.")

Q-values, returns_sum, and returns_count initialized.


In [21]:
def epsilon_greedy_policy(Q, observation, epsilon):
    # Get the number of actions from the environment's action space
    num_actions = env.action_space.n

    # With probability epsilon, choose a random action (exploration)
    if np.random.rand() < epsilon:
        return np.random.randint(num_actions)
    # With probability (1 - epsilon), choose the greedy action (exploitation)
    else:
        # If Q[observation] is not yet defined, it will return an array of zeros
        # np.argmax will pick the first zero if all actions have 0 value
        return np.argmax(Q[observation])
print("Epsilon-greedy policy function defined.")

Epsilon-greedy policy function defined.


In [22]:
def mc_on_policy_control(env, num_episodes, epsilon_start, epsilon_end, epsilon_decay):
    # Initialize Q-values, returns_sum, and returns_count
    Q = defaultdict(lambda: np.zeros(env.action_space.n))
    returns_sum = defaultdict(lambda: np.zeros(env.action_space.n))
    returns_count = defaultdict(lambda: np.zeros(env.action_space.n))

    for i_episode in range(1, num_episodes + 1):
        if i_episode % 10000 == 0:
            print(f"\rEpisode {i_episode}/{num_episodes}", end="")

        # Decay epsilon over episodes
        epsilon = max(epsilon_end, epsilon_start - epsilon_decay * i_episode)

        episode = []
        observation, info = env.reset()
        terminated = False
        truncated = False

        while not terminated and not truncated:
            action = epsilon_greedy_policy(Q, observation, epsilon)
            next_observation, reward, terminated, truncated, info = env.step(action)
            episode.append((observation, action, reward))
            observation = next_observation

        # Convert episode to arrays for easier processing
        states, actions, rewards = zip(*episode)

        # Calculate G (return) for each step in the episode
        # First-visit MC approach: only consider the first time a state-action pair is visited
        sa_in_episode = set()
        for i, (state, action) in enumerate(zip(states, actions)):
            if (state, action) not in sa_in_episode:
                sa_in_episode.add((state, action))
                G = sum(rewards[i:]) # Sum of rewards from this point to the end
                returns_sum[state][action] += G
                returns_count[state][action] += 1.0
                Q[state][action] = returns_sum[state][action] / returns_count[state][action]

    return Q

print("Monte Carlo On-Policy Control function defined.")

Monte Carlo On-Policy Control function defined.


In [23]:
num_episodes_mc = 50000
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = (epsilon_start - epsilon_end) / num_episodes_mc

print(f"Running Monte Carlo On-Policy Control for {num_episodes_mc} episodes...")
Q_optimal = mc_on_policy_control(env, num_episodes_mc, epsilon_start, epsilon_end, epsilon_decay)
print("\nMonte Carlo On-Policy Control completed.")

print("\nEstimated Optimal Action-Value Function (Q):")

# Helper function to get action name
def get_action_name(action):
    return "Stick" if action == 0 else "Hit"

# Print Q-values for a few interesting state-action pairs
# Example: (player_score, dealer_card_showing, usable_ace)

# State: Player has 18, Dealer has 7, No Usable Ace
state1 = (18, 7, False)
if state1 in Q_optimal:
    print(f"State: {state1}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state1][action]:.4f}")
else:
    print(f"State: {state1} not visited.")

# State: Player has 12, Dealer has 2, No Usable Ace
state2 = (12, 2, False)
if state2 in Q_optimal:
    print(f"State: {state2}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state2][action]:.4f}")
else:
    print(f"State: {state2} not visited.")

# State: Player has 20, Dealer has 5, No Usable Ace
state3 = (20, 5, False)
if state3 in Q_optimal:
    print(f"State: {state3}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state3][action]:.4f}")
else:
    print(f"State: {state3} not visited.")

# State: Player has 17, Dealer has Ace, Usable Ace
state4 = (17, 1, True) # Corresponds to (A,6) and dealer showing Ace
if state4 in Q_optimal:
    print(f"State: {state4}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state4][action]:.4f}")
else:
    print(f"State: {state4} not visited.")

# State: Player has 19, Dealer has 8, Usable Ace
state5 = (19, 8, True) # Corresponds to (A,8) and dealer showing 8
if state5 in Q_optimal:
    print(f"State: {state5}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state5][action]:.4f}")
else:
    print(f"State: {state5} not visited.")

Running Monte Carlo On-Policy Control for 50000 episodes...
Episode 50000/50000
Monte Carlo On-Policy Control completed.

Estimated Optimal Action-Value Function (Q):
State: (18, 7, False)
  Action: Stick, Q-Value: 0.3594
  Action: Hit, Q-Value: -0.7130
State: (12, 2, False)
  Action: Stick, Q-Value: -0.3224
  Action: Hit, Q-Value: -0.4848
State: (20, 5, False)
  Action: Stick, Q-Value: 0.6748
  Action: Hit, Q-Value: -0.8740
State: (17, 1, True)
  Action: Stick, Q-Value: -0.8235
  Action: Hit, Q-Value: -0.5405
State: (19, 8, True)
  Action: Stick, Q-Value: 0.6757
  Action: Hit, Q-Value: -0.3500


In [24]:
def sample_policy(observation):
    score, dealer_score, usable_ace = observation
    # Stick if score is 20 or more, else hit
    return 0 if score >= 20 else 1

print("Sample policy defined.")

Sample policy defined.


In [25]:
def generate_episode(env, policy):
    episode = []
    observation, info = env.reset()
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = policy(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        episode.append((observation, action, reward))
        observation = next_observation
    return episode

print("Episode generation function defined.")

Episode generation function defined.


In [26]:
def first_visit_mc_prediction(env, policy, num_episodes):
    # Initialize an empty dictionary for storing returns for each state
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)
    # Initialize an empty dictionary for the state-value function
    V = defaultdict(float)

    for i_episode in range(1, num_episodes + 1):
        if i_episode % 100000 == 0:
            print(f"\rEpisode {i_episode}/{num_episodes}", end="")

        episode = generate_episode(env, policy)
        states, actions, rewards = zip(*episode)

        # Keep track of states visited in this episode
        states_in_episode = set()
        for i, state in enumerate(states):
            if state not in states_in_episode:
                states_in_episode.add(state)
                # Calculate the return G for the first visit to this state
                G = sum(rewards[i:])
                returns_sum[state] += G
                returns_count[state] += 1.0
                V[state] = returns_sum[state] / returns_count[state]
    return V

print("First-Visit MC Prediction function defined.")

First-Visit MC Prediction function defined.


In [27]:
num_episodes = 500000
print(f"Running First-Visit MC Prediction for {num_episodes} episodes...")
V = first_visit_mc_prediction(env, sample_policy, num_episodes)
print("\nMC Prediction completed.")

Running First-Visit MC Prediction for 500000 episodes...
Episode 500000/500000
MC Prediction completed.


In [28]:
print("\nEstimated State-Value Function (V):\n")

sorted_states = sorted(V.keys(), key=lambda x: (x[0], x[1], x[2]))

print("Player Score 12, Dealer Card 2, No Usable Ace:", V.get((12, 2, False), 'N/A'))
print("Player Score 18, Dealer Card 7, No Usable Ace:", V.get((18, 7, False), 'N/A'))
print("Player Score 19, Dealer Card 3, No Usable Ace:", V.get((19, 3, False), 'N/A'))
print("Player Score 20, Dealer Card 5, No Usable Ace:", V.get((20, 5, False), 'N/A'))
print("Player Score 21, Dealer Card 10, No Usable Ace:", V.get((21, 10, False), 'N/A'))
print("Player Score 17, Dealer Card 5, Usable Ace (A,6):", V.get((17, 5, True), 'N/A'))



Estimated State-Value Function (V):

Player Score 12, Dealer Card 2, No Usable Ace: -0.5647425897035881
Player Score 18, Dealer Card 7, No Usable Ace: -0.6828316610925307
Player Score 19, Dealer Card 3, No Usable Ace: -0.7223719676549866
Player Score 20, Dealer Card 5, No Usable Ace: 0.668201516793066
Player Score 21, Dealer Card 10, No Usable Ace: 0.8833203833203833
Player Score 17, Dealer Card 5, Usable Ace (A,6): -0.3477051460361613


In [11]:
num_episodes_mc = 50000
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = (epsilon_start - epsilon_end) / num_episodes_mc

print(f"Running Monte Carlo On-Policy Control for {num_episodes_mc} episodes...")
Q_optimal = mc_on_policy_control(env, num_episodes_mc, epsilon_start, epsilon_end, epsilon_decay)
print("\nMonte Carlo On-Policy Control completed.")

print("\nEstimated Optimal Action-Value Function (Q):")

def get_action_name(action):
    return "Stick" if action == 0 else "Hit"

state1 = (18, 7, False)
if state1 in Q_optimal:
    print(f"State: {state1}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state1][action]:.4f}")
else:
    print(f"State: {state1} not visited.")

state2 = (12, 2, False)
if state2 in Q_optimal:
    print(f"State: {state2}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state2][action]:.4f}")
else:
    print(f"State: {state2} not visited.")

state3 = (20, 5, False)
if state3 in Q_optimal:
    print(f"State: {state3}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state3][action]:.4f}")
else:
    print(f"State: {state3} not visited.")

state4 = (17, 1, True)
if state4 in Q_optimal:
    print(f"State: {state4}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state4][action]:.4f}")
else:
    print(f"State: {state4} not visited.")

state5 = (19, 8, True)
if state5 in Q_optimal:
    print(f"State: {state5}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state5][action]:.4f}")
else:
    print(f"State: {state5} not visited.")

Running Monte Carlo On-Policy Control for 50000 episodes...
Episode 50000/50000
Monte Carlo On-Policy Control completed.

Estimated Optimal Action-Value Function (Q):
State: (18, 7, False)
  Action: Stick, Q-Value: 0.4190
  Action: Hit, Q-Value: -0.8415
State: (12, 2, False)
  Action: Stick, Q-Value: -0.2515
  Action: Hit, Q-Value: -0.4035
State: (20, 5, False)
  Action: Stick, Q-Value: 0.6439
  Action: Hit, Q-Value: -0.9259
State: (17, 1, True)
  Action: Stick, Q-Value: -0.6667
  Action: Hit, Q-Value: -0.4200
State: (19, 8, True)
  Action: Stick, Q-Value: 0.7917
  Action: Hit, Q-Value: -0.4286


In [29]:
num_episodes_mc = 50000
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = (epsilon_start - epsilon_end) / num_episodes_mc

print(f"Running Monte Carlo On-Policy Control for {num_episodes_mc} episodes...")
Q_optimal = mc_on_policy_control(env, num_episodes_mc, epsilon_start, epsilon_end, epsilon_decay)
print("\nMonte Carlo On-Policy Control completed.")

print("\nEstimated Optimal Action-Value Function (Q):\n")

def get_action_name(action):
    return "Stick" if action == 0 else "Hit"

state1 = (18, 7, False)
if state1 in Q_optimal:
    print(f"State: {state1}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state1][action]:.4f}")
else:
    print(f"State: {state1} not visited.")

state2 = (12, 2, False)
if state2 in Q_optimal:
    print(f"State: {state2}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state2][action]:.4f}")
else:
    print(f"State: {state2} not visited.")

state3 = (20, 5, False)
if state3 in Q_optimal:
    print(f"State: {state3}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state3][action]:.4f}")
else:
    print(f"State: {state3} not visited.")
state4 = (17, 1, True)
if state4 in Q_optimal:
    print(f"State: {state4}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state4][action]:.4f}")
else:
    print(f"State: {state4} not visited.")

state5 = (19, 8, True)
if state5 in Q_optimal:
    print(f"State: {state5}")
    for action in range(env.action_space.n):
        print(f"  Action: {get_action_name(action)}, Q-Value: {Q_optimal[state5][action]:.4f}")
else:
    print(f"State: {state5} not visited.")

Running Monte Carlo On-Policy Control for 50000 episodes...
Episode 50000/50000
Monte Carlo On-Policy Control completed.

Estimated Optimal Action-Value Function (Q):

State: (18, 7, False)
  Action: Stick, Q-Value: 0.3852
  Action: Hit, Q-Value: -0.7978
State: (12, 2, False)
  Action: Stick, Q-Value: -0.3640
  Action: Hit, Q-Value: -0.4717
State: (20, 5, False)
  Action: Stick, Q-Value: 0.6750
  Action: Hit, Q-Value: -0.9403
State: (17, 1, True)
  Action: Stick, Q-Value: -0.8571
  Action: Hit, Q-Value: -0.4054
State: (19, 8, True)
  Action: Stick, Q-Value: 0.6098
  Action: Hit, Q-Value: -0.2353
